# Nightly SQM Measurements
**GOAL:**
<br> Plot nightly SQM measurements:
- *x-axis* = Time [sunset -> sunrise]
- *y-axis* = SQM 'mags'
- Filtere the data by cloudcover, lunar phase.

## 1) Read the data

In [1]:
import sqm_read2020
# import sqm_plot2020
import ars_read2020
import numpy as np
import pandas as pd
import sys
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import plotly.express as px
import plotly.graph_objects as go

fname = '/Users/jacksontobin/Local_Documents/NightTime_Research/FoCo Night Sky Team/Fort Collins SQM Monitoring 2020/Fort Collins SQM 2020 data(all_sites_copy2).csv'

In [2]:
# Read the data without filtering the lunar phase or cloud cover
dataframes = sqm_read2020.read_2020(fname=fname, filter_phase=False, filter_cloud=False)

# Read the ARS data 
date_arr = dataframes[0].index
# Riverbend, soapstone, fossil creek, fcmod.
df_ben, df_sop, df_fos, df_dis = ars_read2020.ars_read2020(date_arr=date_arr)


## Initialize df_sqm

In [3]:
# Now we can plot the data by filtering out each lunar phase and cloud cover
# Let's just look at one location:
# 0: FRNA (HZ), 1: FRNA (ZN), 2: FCMOD, 3: PFA1, 4: PFA2, 5: HILT, 6: SPZN, 7: SPHZ
loc = 6
df_sqm = dataframes[loc]

# Choose the dates
start = '2020-06-01'
end   = '2020-06-30'


In [4]:

if loc==6:
    location='SPZN'
elif loc==1:
    location='FRNA'
elif loc==2:
    location='FCMOD'
elif loc==3:
    location='PFA1'
elif loc==4:
    location='PFA2'
elif loc==5:
    location='HILT'
elif loc==7:
    location='SPHZ'
else:
    sys.exit()

## 2) Clean the data

In [5]:
# Clean up the lunar_alt column: some rows are lists, some are floats!?
df_sqm['lunar_alt'] = df_sqm['lunar_alt'].apply(
    lambda x: x[0] if isinstance(x, list) else x)
df_sqm['lunarphaseclass'] = df_sqm['lunarphaseclass'].apply(
    lambda x: x[0] if isinstance(x, list) else x)
df_sqm['mags'] = df_sqm['mags'].apply(
    lambda x: x[0] if isinstance(x, list) else x)
df_sqm['CC'] = df_sqm['CC'].apply(
    lambda x: x[0] if isinstance(x, list) else x)

## 3) Sort data by day?

### Filter by Lunar and Cloud data

In [6]:
# Filter out these lunar phases:
phases = ['Full', 'Gibbous', 'Crescent', 'Quarter']
alts = [-5, -5, -5, -5]
df_sqm = sqm_read2020.filter_sqm_lunar(df_sqm, lunar_phase=phases, lunar_alt=alts)

# # Filter out cloud coverage
coverages = ['OVC', 'SCT', 'FEW', 'BKN', 'M']
df_sqm = sqm_read2020.filter_sqm_cloud(df_sqm, coverage=coverages)

### Sort DataFrame by nights

In [7]:
# Convert the index to DateTime objects
df_sqm.index = pd.to_datetime(df_sqm.index)

# Convert to MST
try:
    df_sqm.index = df_sqm.index.tz_localize('UTC').tz_convert('US/Mountain')
except:
    print('error converting datetime')
    
# Create a new column: all 8pm-6am periods fall on the same date
df_sqm['night'] = (df_sqm.index - pd.Timedelta(hours=20)).normalize()
hours = df_sqm.index.hour

# Filter only nighttime hours (8pm-6am)
# Create the "night" sqm dataset
t_start = 20 # 8pm
t_end = 6 # 6am
df_night = df_sqm[((hours >= t_start) | (hours < t_end)) & df_sqm['mags'].notna()].copy()



### Sort valid dates and assign colors

In [8]:

# Filter to a date range 
#   – Check sqm2020_interactive for reasonable data date ranges!
df_night = df_night[(df_night['night'] >= start) & (df_night['night'] <= end)]

# Convert 'mags' to candela per square meter
df_night['candela'] = 108000*10**(-0.4*df_night['mags'])

# Get the Timestamp of nights
nights = sorted(df_night['night'].unique()) # list of sorted nights (Timestamps)

# Assign colors for each night
cmap = cm.brg
colors = cmap(np.linspace(0, 1, len(nights)))
night_color = {night: color for night, color in zip(nights, colors)}

### Plot the data

In [12]:
from numpy.polynomial import polynomial as P

plot_bestfit = False
plot_scatter = True

fig = go.Figure()

for night, group in df_night.groupby('night'):
    
    # Use fractional hours for x-axis, wrapping past-midnight times
    hours = group.index.hour + group.index.minute / 60
    hours = np.where(hours < t_start, hours + 24, hours)  # 1am -> 25, etc

    night_str = night.strftime('%Y-%m-%d')

    if plot_scatter:
        print(night_color[night])
        fig.add_trace(
            go.Scatter(
                x=hours, 
                y=group['candela'], 
                name=night_str, 
                mode='markers',
                marker=dict(
                    size=8,
                    # color=night_color[night]
                ),
                ))
        # ax.scatter(hours, group['candela'], color=night_color[night],
        #         label=str(night.date()), alpha=0.8, s=4)
    
    # Add a best-fit line with variance:
    if plot_bestfit:
        if len(hours) < 6:
            continue

        # Sort by hour for a clean line
        sort_idx = np.argsort(hours)
        hours_sorted = hours[sort_idx]
        mags_sorted = group['candela'][sort_idx]

        # Fit 5th degree polynomial
        coeffs = np.polyfit(hours_sorted, mags_sorted, 4)
        poly = np.poly1d(coeffs)

        # Evaluate on a smooth grid
        x_grid = np.linspace(hours_sorted.min(), hours_sorted.max(), 300)
        y_fit = poly(x_grid)

        # Residual std dev as variance estimate
        y_at_data = poly(hours_sorted)
        residuals = mags_sorted - y_at_data
        std = np.std(residuals)

        color = night_color[night]
        ax.plot(x_grid, y_fit, color=color, linewidth=1.5, label=str(night.date()))
        ax.fill_between(x_grid, y_fit - std, y_fit + std, color=color, alpha=0.15)

    
# ax.set_xticks(     [20,   21,    22,    23,    24,    25,   26,   27,   28,   29,   30])
# ax.set_xticklabels(['8pm','9pm','10pm','11pm','12am','1am','2am','3am','4am','5am','6am'])
# ax.set_xlabel('')
# ax.set_ylabel('mags (mags/m²)')
# ax.set_ylabel('Cd/m^2')
# ax.set_ylim(0.00, 0.007)
# ax.grid(True, linestyle='--', alpha=0.5)
# ax.set_title(f'{location} Zenith SQM Measurements {start} to {end}')
# ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, markerscale=2)
# plt.tight_layout()
fig.update_layout(
    title=dict(
        text=f'SQM Measurements: {location}, {start} to {end}'
    ),
    xaxis=dict(
        title=dict(text='Hour'),
        tickmode='array',
        tickvals=[20,21, 22, 23, 24, 25,26,27,28,29,30],
        ticktext=['8pm','9pm','10pm','11pm','12am','1am','2am','3am','4am','5am','6am']
    ),
    yaxis=dict(
        title=dict(text='Cd/m^2')
    ),
)

fig.show()
fig.write_html(f'./{location}_{start}_{end}_interactive.html')
# plt.savefig(f'./{location}_{start}-{end}.png')

[0. 0. 1. 1.]
[0.09411765 0.         0.90588235 1.        ]
[0.19607843 0.         0.80392157 1.        ]
[0.29803922 0.         0.70196078 1.        ]
[0.4 0.  0.6 1. ]
[0.50196078 0.         0.49803922 1.        ]
[0.59607843 0.         0.40392157 1.        ]
[0.69803922 0.         0.30196078 1.        ]
[0.8 0.  0.2 1. ]
[0.90196078 0.         0.09803922 1.        ]
[0.99607843 0.00392157 0.         1.        ]
[0.90196078 0.09803922 0.         1.        ]
[0.8 0.2 0.  1. ]
[0.69803922 0.30196078 0.         1.        ]
[0.59607843 0.40392157 0.         1.        ]
[0.49411765 0.50588235 0.         1.        ]
[0.4 0.6 0.  1. ]
[0.29803922 0.70196078 0.         1.        ]
[0.19607843 0.80392157 0.         1.        ]
[0.09411765 0.90588235 0.         1.        ]
[0. 1. 0. 1.]
